# Module 2: Component Information and Circuit Mapping

This notebook parses and cross-validates the Bill of Materials (`data/inputs/bom.csv`) and KiCad netlist (`data/inputs/board.net`) using `components.circuit_map`, displaying component references, values, part numbers, categories, and pin-to-net connections.

In [ ]:
from pathlib import Path
import sys
import json

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in candidates if (p / "data/inputs/board.net").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root containing data/inputs/board.net")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from components.circuit_map import build_circuit_map, export_circuit_map

circuit_map = build_circuit_map()
artifacts = export_circuit_map(circuit_map)
print(json.dumps(circuit_map["validation"], indent=2))
print("\nCategory counts:", json.dumps(circuit_map["category_summary"], indent=2))

In [ ]:
# Display active semiconductor and IC components (Transistors, ICs, Regulators) with pin-to-net connections
for comp in circuit_map["components"]:
    if comp["reference"].startswith(("Q", "U")):
        print(f"=== {comp['reference']} | Value: {comp['value']} | Footprint: {comp['footprint']} ===")
        for pin in comp["pins"]:
            peers = ", ".join(pin["connected_peers"][:8])
            if len(pin["connected_peers"]) > 8:
                peers += f" ... (+{len(pin['connected_peers']) - 8} more)"
            print(f"  Pin {pin['pin']:>2} ({pin['pinfunction'] or '-':<14}) [{pin['pintype']:<13}] -> {pin['net_name']} | Peers: {peers}")
        print()